# Fuzy-Jon Package Examples

Fuzy-jon is a Python package for repairing invalid JSON, particularly useful when working with language model outputs that may generate malformed JSON.

In [ ]:
# Example 1: Basic JSON Repair
from fuzzy_json import loads
import json

print("Example 1: Basic JSON Repair")

# Test with valid JSON first
valid_json = '{"name": "Alice", "age": 30, "active": true}'
result = loads(valid_json)
print(f"✓ Valid JSON: {result}")

# Test with JSON errors that fuzzy_json can actually fix
fixable_jsons = [
    "{'name': 'Alice', 'age': 30}",     # Single quotes (works)
    '{"name": "Alice", "age": 30,}',     # Trailing comma (works) 
    '{"name": "Alice", "city": "NYC"}',  # Valid example
]

expected_repairs = [{'name': 'Alice', 'age': 30}, {'name': 'Alice', 'age': 30}, {'name': 'Alice', 'city': 'NYC'}]
print("✓ Repairing fixable JSON:")
for i, (json_str, expected) in enumerate(zip(fixable_jsons, expected_repairs, strict=True)):
    repaired = loads(json_str)
    assert repaired == expected, (json_str, repaired)
    print(f"  {i+1}. ✓ Repaired: {repaired}")

# Show what fuzzy_json cannot fix
print("✓ Examples fuzzy_json cannot repair:")
unfixable = ["invalid text", "", "{'incomplete'"]
for i, json_str in enumerate(unfixable):
    # past repair, fuzzy_json raises the stdlib's JSONDecodeError with a "Failed to repair JSON" message
    try:
        result = loads(json_str)
    except json.JSONDecodeError as e:
        assert "Failed to repair JSON" in str(e), e
        print(f"  {i+1}. ✓ Correctly failed: {json_str}")
    else:
        raise AssertionError(f"fuzzy_json unexpectedly repaired {json_str!r} into {result!r}")

print("✓ Basic JSON repair completed")

## Example 2: Language Model Output Repair

In [ ]:
# Example 2: Language Model Output Repair
from fuzzy_json import loads

print("Example 2: Language Model Output Repair")

# Realistic LLM outputs that fuzzy_json can actually handle
llm_outputs = [
    # Single quotes instead of double quotes
    "{'user': 'Alice', 'role': 'admin'}",
    
    # Trailing commas
    '{"name": "Bob", "score": 95,}',
    
    # Mixed quote styles that might work
    '{"user": "Charlie", "active": true}',
    
    # Array with trailing comma
    '{"tags": ["python", "json", "parsing",]}',
]

expected_repairs = [
    {'user': 'Alice', 'role': 'admin'},
    {'name': 'Bob', 'score': 95},
    {'user': 'Charlie', 'active': True},
    {'tags': ['python', 'json', 'parsing']},
]
print("✓ Processing LLM-style JSON outputs:")
for i, (output, expected) in enumerate(zip(llm_outputs, expected_repairs, strict=True)):
    repaired = loads(output)
    assert repaired == expected, (output, repaired)
    print(f"  {i+1}. ✓ Successfully parsed")
    print(f"      Input: {output}")
    print(f"      Result: {repaired}")

print("✓ Language model output repair completed")

## Example 3: Complex Data Structure Repair

In [ ]:
# Example 3: Complex Data Structure Repair
from fuzzy_json import loads

print("Example 3: Complex Data Structure Repair")

# Complex structures that fuzzy_json can handle
complex_jsons = [
    # Nested objects with trailing commas
    '{"user": {"name": "Alice", "age": 30,}, "active": true}',
    
    # Arrays with trailing commas
    '{"items": [1, 2, 3,], "count": 3}',
    
    # Mixed single and double quotes (if supported)
    "{'config': {'debug': true, 'port': 8080}}",
    
    # Multiple trailing commas
    '{"a": 1, "b": 2,}',
]

expected_repairs = [
    {'user': {'name': 'Alice', 'age': 30}, 'active': True},
    {'items': [1, 2, 3], 'count': 3},
    {'config': {'debug': True, 'port': 8080}},
    {'a': 1, 'b': 2},
]
print("✓ Repairing complex JSON structures:")
for i, (complex_json, expected) in enumerate(zip(complex_jsons, expected_repairs, strict=True)):
    repaired = loads(complex_json)
    assert repaired == expected, (complex_json, repaired)
    print(f"  {i+1}. ✓ Successfully repaired")
    print(f"      Result type: {type(repaired).__name__}")
    if isinstance(repaired, dict):
        print(f"      Keys: {list(repaired.keys())}")

print("✓ Complex data structure repair completed")

## Example 4: Error Handling and Fallbacks

In [ ]:
# Example 4: Error Handling and Fallbacks
from fuzzy_json import loads

print("Example 4: Error Handling and Fallbacks")

def safe_json_parse(json_string, fallback=None):
    """Safely parse JSON with fuzzy repair and fallback."""
    try:
        # Handle None input
        if json_string is None:
            return fallback
        # Try fuzzy JSON repair
        return loads(json_string)
    except Exception as e:
        print(f"    Fuzzy repair failed: {str(e)[:50]}...")
        return fallback

# Test various inputs with proper error handling
test_cases = [
    ('{"valid": "json"}', "Valid JSON"),
    ("{'single': 'quotes'}", "Single quotes"),
    ('{"trailing": "comma",}', "Trailing comma"),
    ('invalid text here', "Non-JSON text"),
    ('', "Empty string"),
    (None, "None input"),
]

print("✓ Testing safe JSON parsing:")
for test_input, description in test_cases:
    print(f"  {description}:")
    result = safe_json_parse(test_input, {"error": "Could not parse"})
    print(f"    Result: {result}")

print("✓ Error handling and fallbacks completed")

## Example 5: Production Use Cases

In [ ]:
# Example 5: Production Use Cases
from fuzzy_json import loads

print("Example 5: Production Use Cases")

# Realistic production scenarios with fixable JSON issues
production_cases = [
    # Configuration with trailing comma
    ('{"api_key": "abc123", "timeout": 30, "retries": 3,}', "Config with trailing comma"),
    
    # Single quotes from Python repr
    ("{'function': 'calculate', 'args': {'x': 5, 'y': 10}}", "Python-style quotes"),
    
    # Array with trailing comma
    ('["item1", "item2", "item3",]', "Array with trailing comma"),
    
    # Nested structure with mixed issues
    ('{"data": {"values": [1, 2, 3,], "total": 6,}}', "Nested trailing commas"),
]

print("✓ Production use case scenarios:")
for i, (json_str, description) in enumerate(production_cases):
    try:
        parsed = loads(json_str)
        print(f"  {i+1}. ✓ {description}")
        if isinstance(parsed, dict):
            print(f"      Dict with {len(parsed)} keys")
        elif isinstance(parsed, list):
            print(f"      List with {len(parsed)} items")
        else:
            print(f"      Type: {type(parsed).__name__}")
    except Exception as e:
        print(f"  {i+1}. ✗ {description}: {str(e)[:40]}...")
        raise

print("✓ Production use cases demonstrated")
print("✓ Fuzy-jon handles common JSON formatting issues")